# SOXX 原始 N-PORT 成分股价格来源门

冻结 `plan.json`：逐行验证原始 XML 股值与旧 ticker 映射、CUSIP；通过 Qlib D.calendar/D.features 一次读取 2019–2022 输入价格，计算 20 日窗口完整性。此为来源诊断，0 目标收益、0 拟合、0 策略账户。

In [1]:
import os,json,hashlib,csv,pickle,tarfile,tempfile,subprocess,shlex,collections
from pathlib import Path
from decimal import Decimal
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder,_settings
root=Path.cwd();card=root/'research/stage1_soxx_nport_member_price_preflight_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists()
for block,pathkey,hashkey in [(plan['original_source'],'card_summary','card_summary_sha256'),(plan['late_training_map'],'summary','summary_sha256'),(plan['legacy_price_provider'],'manifest','manifest_sha256'),(plan['legacy_price_provider'],'ticker_bridge','ticker_bridge_sha256')]:assert sha(root/block[pathkey])==block[hashkey]
original_summary=json.loads((root/plan['original_source']['card_summary']).read_text());map_summary=json.loads((root/plan['late_training_map']['summary']).read_text())
assert original_summary['recorder_id']==plan['original_source']['recorder_id'] and map_summary['recorder_id']==plan['late_training_map']['recorder_id']
state={'status':'started','Qlib_price_feature_batch_reads':0,'Qlib_source_Recorder_object_loads':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/plan['legacy_price_provider']['provider']),region='us',kernels=1,expression_cache=None,dataset_cache=None)
with R.uri_context(original_summary['recorder_uri']):
 rec=R.get_recorder(recorder_id=original_summary['recorder_id'],experiment_id=original_summary['experiment_id']);original_events=rec.load_object('analysis/source_events');original_members=rec.load_object('analysis/member_rows')
state['Qlib_source_Recorder_object_loads']+=2
with R.uri_context(map_summary['recorder_uri']):
 rec=R.get_recorder(recorder_id=map_summary['recorder_id'],experiment_id=map_summary['experiment_id']);mapped=rec.load_object(plan['late_training_map']['artifact'])
state['Qlib_source_Recorder_object_loads']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
assert len(original_events)==13 and len(original_members)==390 and len(mapped)==1008
bridge=list(csv.DictReader((root/plan['legacy_price_provider']['ticker_bridge']).open()))
source_accessions={x['accession'] for x in original_members};members=[];per_report=[]
for accession in sorted(source_accessions):
 original=[x for x in original_members if x['accession']==accession];listed=[x for x in bridge if x['accession']==accession]
 assert len(original)==len(listed)==30 and all(x['series_id']=='S000004354' for x in listed)
 assert collections.Counter(Decimal(x['fair_value_usd_original']) for x in original)==collections.Counter(Decimal(x['market_value_usd']) for x in listed)
 total=sum(Decimal(x['fair_value_usd_original']) for x in original);names=set();tickers=set()
 for item in original:
  value=Decimal(item['fair_value_usd_original']);candidate=[z for z in listed if Decimal(z['market_value_usd'])==value]
  assert len(candidate)==1
  match=candidate[0];assert match['ticker'] and match['ticker'] not in tickers;tickers.add(match['ticker'])
  strict=item['identifier_status']=='original_unique_CUSIP'
  if strict:assert match['cusip']==item['cusip_original']
  else:assert item['cusip_original'] in {'000000000','N/A'}
  assert item['security_original'].casefold() not in names;names.add(item['security_original'].casefold())
  members.append({'accession':accession,'report_period':item['report_period'],'security_original':item['security_original'],'ticker_from_legacy_bridge':match['ticker'],'cusip_original':item['cusip_original'],'strict_CUSIP_identity':strict,'original_value_usd':str(value),'original_equity_weight':float(value/total)})
 per_report.append({'accession':accession,'rows':30,'strict_CUSIP_rows':sum(x['strict_CUSIP_identity'] for x in members if x['accession']==accession),'name_value_only_rows':sum(not x['strict_CUSIP_identity'] for x in members if x['accession']==accession)})
assert len(members)==390 and sum(x['strict_CUSIP_identity'] for x in members)==357 and sum(not x['strict_CUSIP_identity'] for x in members)==33
calendar=pd.DatetimeIndex(D.calendar(start_time='2019-10-01',end_time='2022-12-30',freq='day'))
active=mapped[mapped['form']=='NPORT-P'].copy();assert len(active)==780 and active['accession'].nunique()==13
available=set(D.list_instruments(D.instruments('all'),start_time='2019-10-01',end_time='2022-12-30',as_list=True));tickers=sorted({x['ticker_from_legacy_bridge'] for x in members});to_read=sorted(set(tickers)&available)
state['Qlib_price_feature_batch_reads']=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
prices=D.features(instruments=to_read,fields=['$adj_close'],start_time='2019-10-01',end_time='2022-12-30',freq='day')
pr=prices.reset_index();datecol='datetime' if 'datetime' in pr.columns else 'date';instrumentcol='instrument';valuecol='$adj_close' if '$adj_close' in pr.columns else prices.columns[0]
wide=pr.pivot(index=datecol,columns=instrumentcol,values=valuecol).reindex(index=calendar,columns=tickers)
valid=np.isfinite(wide.to_numpy(dtype=float))&(wide.to_numpy(dtype=float)>0)
complete=pd.DataFrame(valid,index=calendar,columns=tickers).rolling(21,min_periods=21).sum().eq(21)
by_accession=collections.defaultdict(list)
for x in members:by_accession[x['accession']].append(x)
daily=[];missing_ticker_days=collections.Counter();missing_ticker_weight=collections.Counter()
for _,row in active.iterrows():
 day=pd.Timestamp(row['signal_date']);accession=row['accession'];group=by_accession[accession];assert len(group)==30 and day in complete.index
 available20=complete.loc[day];missing=[x for x in group if not bool(available20[x['ticker_from_legacy_bridge']])];missing_weight=sum(x['original_equity_weight'] for x in missing)
 for x in missing:
  missing_ticker_days[x['ticker_from_legacy_bridge']]+=1;missing_ticker_weight[x['ticker_from_legacy_bridge']]+=x['original_equity_weight']
 daily.append({'signal_date':day,'accession':accession,'report_period':row['report_period'],'full_30_price_window':len(missing)==0,'strict_30_identity_and_price_window':len(missing)==0 and all(x['strict_CUSIP_identity'] for x in group),'price_complete_members':30-len(missing),'missing_members':len(missing),'missing_original_equity_weight':missing_weight,'placeholder_identity_members':sum(not x['strict_CUSIP_identity'] for x in group),'missing_tickers':[x['ticker_from_legacy_bridge'] for x in missing]})
daily=pd.DataFrame(daily).sort_values('signal_date').reset_index(drop=True)
assert len(daily)==780 and daily.signal_date.is_unique
calendar_frame=daily.set_index('signal_date')[['price_complete_members','missing_members','missing_original_equity_weight','placeholder_identity_members','full_30_price_window','strict_30_identity_and_price_window']]
calendar_frame.index=pd.MultiIndex.from_arrays([calendar_frame.index,['SOXL_SOXX_CASH']*len(calendar_frame)],names=['datetime','instrument'])
static=pd.concat({'feature':calendar_frame[['price_complete_members','missing_original_equity_weight']],'meta':calendar_frame[['missing_members','placeholder_identity_members','full_30_price_window','strict_30_identity_and_price_window']]},axis=1)
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'nport_late_train':(str(daily.signal_date.min().date()),str(daily.signal_date.max().date()))})
prepared=dataset.prepare('nport_late_train',col_set=['feature','meta']);assert len(prepared)==780
summary_audit={'status':'original_NPORT_2019_2022_member_price_coverage_diagnostic_only','source_plan_sha256':sha(card/'plan.json'),'original_report_events':13,'original_equity_rows':390,'strict_CUSIP_value_matches':357,'placeholder_name_value_only_rows':33,'legacy_ticker_bridge_rows_matched':390,'NPORT_active_Qlib_sessions':780,'distinct_bridge_tickers':len(tickers),'Qlib_provider_tickers_present':len(to_read),'Qlib_provider_tickers_absent':sorted(set(tickers)-available),'full_30_member_21_close_window_sessions':int(daily.full_30_price_window.sum()),'strict_full_30_identity_and_price_window_sessions':int(daily.strict_30_identity_and_price_window.sum()),'median_missing_members':float(daily.missing_members.median()),'max_missing_original_equity_weight':float(daily.missing_original_equity_weight.max()),'median_missing_original_equity_weight':float(daily.missing_original_equity_weight.median()),'top_missing_ticker_days':missing_ticker_days.most_common(10),'source_price_vintage_qualified':False,'2011_2022_full_Stage1_feature_qualified':False,'new_GETs':0,'Qlib_price_feature_batch_reads':1,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'source_audit.json').write_text(json.dumps(summary_audit,indent=2)+'\n')
dataset.config(dump_all=True,recursive=True);assert len(pickle.loads(pickle.dumps(dataset)).prepare('nport_late_train',col_set=['feature','meta']))==780
tracking=card/'qlib_recorder'
with R.start(experiment_name='SOXX_NPORT_member_price_preflight',uri=str(tracking)):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_hashes={'original':plan['original_source']['card_summary_sha256'],'map':plan['late_training_map']['summary_sha256'],'price_provider':plan['legacy_price_provider']['manifest_sha256'],'ticker_bridge':plan['legacy_price_provider']['ticker_bridge_sha256']})
 rec.save_objects(artifact_path='analysis',matched_members=members,per_report=per_report,daily_coverage=daily,source_audit=summary_audit)
 rec.save_objects(artifact_path='metadata',dataset=dataset)
 rec.log_metrics(original_equity_rows=390,NPORT_active_Qlib_sessions=780,full_30_price_sessions=int(daily.full_30_price_window.sum()),model_fits=0,native_accounts=0)
 recorder_id=rec.id;experiment_id=str(rec.experiment_id)
with R.uri_context(str(tracking)):
 checked=R.get_recorder(recorder_id=recorder_id,experiment_id=experiment_id);assert len(checked.load_object('analysis/daily_coverage'))==780 and len(checked.load_object('metadata/dataset').prepare('nport_late_train',col_set=['feature','meta']))==780
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':'SOXX_NPORT_member_price_preflight','ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=recorder_id,experiment_id=experiment_id)
with tempfile.TemporaryDirectory(prefix='soxx-price-preflight-restore-') as temp:
 archive=Path(temp)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as source:names=[m.name for m in source.getmembers() if m.isfile()]
 assert any(name.endswith('/artifacts/metadata/dataset') for name in names)
summary={'experiment':card.name,'decision':'NPORT_member_price_source_coverage_diagnostic_no_Stage1_feature_or_account','objective_achieved':False,'plan':'plan.json','source_audit':'source_audit.json','model_fits':0,'native_accounts':0,'recorder_id':recorder_id,'experiment_id':experiment_id,'recorder_uri':str(tracking),'nas_evidence':evidence,'nas_readback':{'sha256':evidence['sha256'],'verified':True},'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/*','metadata/*']}}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
state['status']='completed';state['recorder_id']=recorder_id;state['nas_archive_sha256']=evidence['sha256'];(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
print({k:summary_audit[k] for k in ('NPORT_active_Qlib_sessions','distinct_bridge_tickers','Qlib_provider_tickers_absent','full_30_member_21_close_window_sessions','strict_full_30_identity_and_price_window_sessions','median_missing_members','median_missing_original_equity_weight','top_missing_ticker_days')})
print({'recorder_id':recorder_id,'nas_sha256':evidence['sha256']})


{'NPORT_active_Qlib_sessions': 780, 'distinct_bridge_tickers': 39, 'Qlib_provider_tickers_absent': ['BRKS', 'CCMP', 'CREE', 'IPHI', 'MXIM', 'WOLF', 'XLNX'], 'full_30_member_21_close_window_sessions': 0, 'strict_full_30_identity_and_price_window_sessions': 0, 'median_missing_members': 4.0, 'median_missing_original_equity_weight': 0.05695845070007403, 'top_missing_ticker_days': [('XLNX', 630), ('CREE', 567), ('CCMP', 441), ('MXIM', 316), ('WOLF', 213), ('BRKS', 188), ('IPHI', 188)]}
{'recorder_id': '073fd00eeb594f82be3741ad091091c6', 'nas_sha256': 'acc9cac4842d3cbd84af121d50f38d73b22c6daef409a1d64cf3c80c196db51d'}


## 中文结论

13 份原始 N-PORT 的 390 条市值与旧 ticker 桥全部对齐；357 条具原始 CUSIP 精确一致，33 条占位 CUSIP 仅同报告名称/市值匹配。既有 Qlib provider 在 39 个所需 ticker 中缺 BRKS、CCMP、CREE、IPHI、MXIM、WOLF、XLNX；780 个 N-PORT 生效日中完整 30 股的信号日及前 20 日价格窗口为 0。Qlib DatasetH/Recorder `073fd00eeb594f82be3741ad091091c6` 和 NAS SHA-256 `acc9cac4842d3cbd84af121d50f38d73b22c6daef409a1d64cf3c80c196db51d` 回读通过。0 SOXX 收益/标签、0 拟合、0 账户。当前来源不能支持预定新低扩散/下跌集中度信息拟合；工作区有 WOLF/CREE/BRKS 补充来源但不能把单一 provider 缺口说成全局缺失。